# TRELLIS.2 Symmetry Projection Guidance

This notebook runs TRELLIS.2 with SymTRELLIS symmetry projection guidance. It keeps the original generation pipeline explicit: image conditioning, sparse-structure flow, sparse-structure SPG, shape-latent flow, shape-latent SPG, and mesh decoding.

Expected input image path: `examples/input.png`.

In [ ]:
import gc
import os
import sys
from pathlib import Path

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "symtrellis").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "third_party" / "trellis2"))

import cumesh
import numpy as np
import torch
import trimesh
from PIL import Image

import trellis2.models as models
from trellis2.modules.image_feature_extractor import DinoV3FeatureExtractor
from trellis2.modules.sparse import SparseTensor
from trellis2.pipelines.rembg import BiRefNet

from inference.trellis2 import (
    TRELLIS2_SHAPE_LATENT_CFG_INTERVAL,
    TRELLIS2_SHAPE_LATENT_CFG_RESCALE,
    TRELLIS2_SHAPE_LATENT_CFG_STRENGTH,
    TRELLIS2_SHAPE_LATENT_RESCALE_T,
    TRELLIS2_SHAPE_LATENT_STEPS,
    TRELLIS2_SPARSE_STRUCTURE_CFG_INTERVAL,
    TRELLIS2_SPARSE_STRUCTURE_CFG_RESCALE,
    TRELLIS2_SPARSE_STRUCTURE_CFG_STRENGTH,
    TRELLIS2_SPARSE_STRUCTURE_RESCALE_T,
    TRELLIS2_SPARSE_STRUCTURE_STEPS,
    TRELLIS2FlowPredictor,
    TRELLIS2ShapeLatentView,
    TRELLIS2ShapeLatentNoiseSampler,
    TRELLIS2SparseStructureLatentNoiseSampler,
    TRELLIS2SparseStructureView,
    trellis2_dense_grid_coords,
    trellis2_sparse_structure_logits_to_coords,
)
from symtrellis.flow import (
    ClassifierFreeGuidanceWrapper,
    EulerSolver,
    SymmetryProjectionGuidanceWrapper,
    SymmetryProjectionNoiseSampler,
)
from symtrellis.mapper import Swin3DLatentMapper, Swin3DLatentMapperConfig, SymmetryProjector
from symtrellis.symmetry import build_symmetry_relation_inputs, get_3d_point_group

In [ ]:
DEVICE = "cuda:0"
SIGMA_MIN = 1e-5
SEED = 42

EXAMPLES_DIR = REPO_ROOT / "examples"
IMAGE_PATH = EXAMPLES_DIR / "input.png"
OUTPUT_GLB_PATH = EXAMPLES_DIR / "symm_enforce_trellis2.glb"

SS_FLOW_MODEL_PATH = "microsoft/TRELLIS.2-4B/ckpts/ss_flow_img_dit_1_3B_64_bf16"
SS_DECODER_PATH = "microsoft/TRELLIS-image-large/ckpts/ss_dec_conv3d_16l8_fp16"
SHAPE_FLOW_MODEL_PATH = "microsoft/TRELLIS.2-4B/ckpts/slat_flow_img2shape_dit_1_3B_512_bf16"
SHAPE_DECODER_PATH = "microsoft/TRELLIS.2-4B/ckpts/shape_dec_next_dc_f16c32_fp16"
IMAGE_COND_MODEL_NAME = "facebook/dinov3-vitl16-pretrain-lvd1689m"
REMBG_MODEL_NAME = "briaai/RMBG-2.0"

SPARSE_STRUCTURE_MAPPER_CKPT = REPO_ROOT / "checkpoints" / "trellis2_sparse_structure_swin3d_latent_mapper_base.pt"
SHAPE_LATENT_MAPPER_CKPT = REPO_ROOT / "checkpoints" / "trellis2_shape_latent_swin3d_latent_mapper_small.pt"

SS_FEAT_DIM = 8
SS_LATENT_RESOLUTION = 16
SS_TARGET_RESOLUTION = 32

SHAPE_FEAT_DIM = 32
SHAPE_LATENT_RESOLUTION = 32
SHAPE_RESOLUTION = 512

SPARSE_STRUCTURE_NOISE_SYMMETRY_STRENGTH = 0.2
SPARSE_STRUCTURE_SPG_STRENGTH = 0.4
SPARSE_STRUCTURE_SPG_INTERVAL = (0.0, 0.3)

SHAPE_LATENT_NOISE_SYMMETRY_STRENGTH = 0.0
SHAPE_LATENT_SPG_STRENGTH = 0.0
SHAPE_LATENT_SPG_INTERVAL = (0.0, 0.3)

In [ ]:
def preprocess_image(
    image: Image.Image,
    rembg_model,
    target_size: int = 512,
    extend_scale: float = 1.05,
) -> Image.Image:
    has_alpha = False
    if image.mode == "RGBA":
        alpha = np.array(image)[:, :, 3]
        if not np.all(alpha == 255):
            has_alpha = True

    if has_alpha:
        processed_image = image
    else:
        image = image.convert("RGB")
        processed_image = rembg_model(image)

    processed_image_np = np.array(processed_image)
    alpha = processed_image_np[:, :, 3]

    bbox = np.argwhere(alpha > 0.8 * 255)
    bbox = np.min(bbox[:, 1]), np.min(bbox[:, 0]), np.max(bbox[:, 1]), np.max(bbox[:, 0])
    center = float((bbox[0] + bbox[2]) / 2), float((bbox[1] + bbox[3]) / 2)
    size = max(int(bbox[2] - bbox[0]), int(bbox[3] - bbox[1]))
    size = int(size * extend_scale)
    bbox = center[0] - size // 2, center[1] - size // 2, center[0] + size // 2, center[1] + size // 2

    processed_image = processed_image.crop(bbox)
    max_size = max(image.size)
    scale = min(1, target_size / max_size)
    if scale < 1:
        processed_image = processed_image.resize((int(image.width * scale), int(image.height * scale)), Image.Resampling.LANCZOS)
    processed_image = np.array(processed_image).astype(np.float32) / 255
    processed_image = processed_image[:, :, :3] * processed_image[:, :, 3:4]
    processed_image = Image.fromarray((processed_image * 255).astype(np.uint8))

    return processed_image


def voxel_grid_to_cube_mesh(vox_bool, grid_res: int = 64):
    occupied = np.argwhere(vox_bool)
    print(f"Building {len(occupied)} cubes ...")

    if len(occupied) == 0:
        return trimesh.Trimesh(
            vertices=np.zeros((0, 3)),
            faces=np.zeros((0, 3), dtype=np.int64),
        )

    cube_size = 1.0 / grid_res
    cube_template = trimesh.creation.box(extents=(cube_size, cube_size, cube_size))

    cubes = []
    for vox_id in occupied:
        center = (vox_id.astype(np.float32) + 0.5) / grid_res - 0.5
        cube = cube_template.copy()
        cube.apply_translation(center)
        cubes.append(cube)

    return trimesh.util.concatenate(cubes)


def clean_trellis2_mesh_output(
    shape_mesh,
    res: int,
    remesh: bool = True,
    simplify: bool = True,
    decimation_target: int = 500000,
):
    vertices = shape_mesh.vertices.detach().contiguous().float().cuda()
    faces = shape_mesh.faces.detach().contiguous().int().cuda()

    mesh = cumesh.CuMesh()
    mesh.init(vertices, faces)

    del vertices, faces
    gc.collect()
    torch.cuda.empty_cache()

    mesh.fill_holes(max_hole_perimeter=3e-2)

    if remesh:
        vertices, faces = mesh.read()
        bvh = cumesh.cuBVH(vertices, faces)

        center = torch.tensor([0.0, 0.0, 0.0]).cuda()
        scale = 1.0
        resolution = int(res)

        remesh_vertices, remesh_faces = cumesh.remeshing.remesh_narrow_band_dc(
            vertices,
            faces,
            center=center,
            scale=(resolution + 3) / resolution * scale,
            resolution=resolution,
            band=1,
            project_back=0.9,
            verbose=False,
            bvh=bvh,
        )

        del bvh, vertices, faces, mesh
        gc.collect()
        torch.cuda.empty_cache()

        mesh = cumesh.CuMesh()
        mesh.init(remesh_vertices, remesh_faces)
        del remesh_vertices, remesh_faces
        gc.collect()
        torch.cuda.empty_cache()

        if simplify:
            mesh.simplify(decimation_target, verbose=False)

    mesh.compute_vertex_normals()
    out_vertices, out_faces = mesh.read()
    out_normals = mesh.read_vertex_normals()

    vertices_np = out_vertices.cpu().numpy().copy()
    faces_np = out_faces.cpu().numpy()
    normals_np = out_normals.cpu().numpy().copy()

    vy = vertices_np[:, 1].copy()
    vz = vertices_np[:, 2].copy()
    vertices_np[:, 1] = vz
    vertices_np[:, 2] = -vy

    ny = normals_np[:, 1].copy()
    nz = normals_np[:, 2].copy()
    normals_np[:, 1] = nz
    normals_np[:, 2] = -ny

    glb = trimesh.Trimesh(
        vertices=vertices_np,
        faces=faces_np,
        vertex_normals=normals_np,
        process=False,
    )

    return glb, out_vertices, out_faces

In [ ]:
ss_flow_model = models.from_pretrained(SS_FLOW_MODEL_PATH).eval()
ss_decoder = models.from_pretrained(SS_DECODER_PATH).eval()

shape_flow_model = models.from_pretrained(SHAPE_FLOW_MODEL_PATH).eval()
shape_decoder = models.from_pretrained(SHAPE_DECODER_PATH).eval()
shape_decoder.set_resolution(SHAPE_RESOLUTION)

image_cond_model = DinoV3FeatureExtractor(IMAGE_COND_MODEL_NAME, image_size=512)
rembg_model = BiRefNet(REMBG_MODEL_NAME)

In [ ]:
ss_mapper_ckpt = torch.load(SPARSE_STRUCTURE_MAPPER_CKPT, map_location="cpu")
ss_mapper_config = dict(ss_mapper_ckpt["config"])
ss_mapper_config["attn_backend"] = "flash_attn"
ss_mapper = Swin3DLatentMapper(Swin3DLatentMapperConfig(**ss_mapper_config))
ss_mapper.load_state_dict(ss_mapper_ckpt["model"], strict=True)
ss_mapper.eval()

shape_mapper_ckpt = torch.load(SHAPE_LATENT_MAPPER_CKPT, map_location="cpu")
shape_mapper_config = dict(shape_mapper_ckpt["config"])
shape_mapper_config["attn_backend"] = "flash_attn"
shape_mapper = Swin3DLatentMapper(Swin3DLatentMapperConfig(**shape_mapper_config))
shape_mapper.load_state_dict(shape_mapper_ckpt["model"], strict=True)
shape_mapper.eval();

In [ ]:
ss_noise_sampler = TRELLIS2SparseStructureLatentNoiseSampler()
ss_flow_predictor = TRELLIS2FlowPredictor(model=ss_flow_model)
ss_flow_cfg_predictor = ClassifierFreeGuidanceWrapper(
    predictor=ss_flow_predictor,
    strength=TRELLIS2_SPARSE_STRUCTURE_CFG_STRENGTH,
    interval=TRELLIS2_SPARSE_STRUCTURE_CFG_INTERVAL,
    rescale=TRELLIS2_SPARSE_STRUCTURE_CFG_RESCALE,
)
ss_flow_spg_predictor = SymmetryProjectionGuidanceWrapper(
    predictor=ss_flow_cfg_predictor,
    strength=SPARSE_STRUCTURE_SPG_STRENGTH,
    interval=SPARSE_STRUCTURE_SPG_INTERVAL,
    symmetrize_target="x_start",
    rescale=0.0,
)
ss_noise_spg_sampler = SymmetryProjectionNoiseSampler(
    sampler=ss_noise_sampler,
    symmetry_strength=SPARSE_STRUCTURE_NOISE_SYMMETRY_STRENGTH,
)

shape_noise_sampler = TRELLIS2ShapeLatentNoiseSampler()
shape_flow_predictor = TRELLIS2FlowPredictor(model=shape_flow_model)
shape_flow_cfg_predictor = ClassifierFreeGuidanceWrapper(
    predictor=shape_flow_predictor,
    strength=TRELLIS2_SHAPE_LATENT_CFG_STRENGTH,
    interval=TRELLIS2_SHAPE_LATENT_CFG_INTERVAL,
    rescale=TRELLIS2_SHAPE_LATENT_CFG_RESCALE,
)
shape_flow_spg_predictor = SymmetryProjectionGuidanceWrapper(
    predictor=shape_flow_cfg_predictor,
    strength=SHAPE_LATENT_SPG_STRENGTH,
    interval=SHAPE_LATENT_SPG_INTERVAL,
    symmetrize_target="x_start",
    rescale=0.0,
)
shape_noise_spg_sampler = SymmetryProjectionNoiseSampler(
    sampler=shape_noise_sampler,
    symmetry_strength=SHAPE_LATENT_NOISE_SYMMETRY_STRENGTH,
)

flow_solver = EulerSolver()

In [ ]:
image = Image.open(IMAGE_PATH)

rembg_model.to(DEVICE)
processed_image = preprocess_image(image, rembg_model=rembg_model)
rembg_model.cpu()

image_cond_model.to(DEVICE)
cond = image_cond_model([processed_image])
neg_cond = torch.zeros_like(cond)
image_cond_model.cpu()

processed_image

In [ ]:
symmetry_label = "C3"
symmetry_center = torch.tensor([0.0, 0.0, 0.0], device=DEVICE)
symmetry_major_axis = torch.tensor([0.0, 0.0, 1.0], device=DEVICE)
symmetry_minor_axis = torch.tensor([1.0, 0.0, 0.0], device=DEVICE)

O_dst2src, t_dst2src, s_dst2src = get_3d_point_group(
    label=symmetry_label,
    center=symmetry_center,
    major_axis=symmetry_major_axis,
    minor_axis=symmetry_minor_axis,
    include_identity=False,
)

relations = [(O_dst2src, t_dst2src, s_dst2src)]
O_dst2src.shape, t_dst2src.shape, s_dst2src.shape

In [ ]:
ss_coords = trellis2_dense_grid_coords(
    batch_size=1,
    grid_size=SS_LATENT_RESOLUTION,
    device=DEVICE,
)
ss_coords_src, ss_coords_dst, ss_rows_src, ss_rows_dst, ss_O, ss_t, ss_s = build_symmetry_relation_inputs(
    coords=ss_coords,
    relations=relations,
    grid_size=SS_LATENT_RESOLUTION,
)

ss_mapper.to(DEVICE)
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    ss_coeff = ss_mapper(
        coords_src=ss_coords_src,
        coords_dst=ss_coords_dst,
        O_dst2src=ss_O,
        t_dst2src=ss_t,
        s_dst2src=ss_s,
    )
ss_coeff = ss_coeff.to(device=torch.device(DEVICE), dtype=torch.float32)
ss_mapper.cpu()

ss_projector = SymmetryProjector(
    num_rows=ss_coords.shape[0],
    rows_src=ss_rows_src,
    rows_dst=ss_rows_dst,
    coeff=ss_coeff,
)

ss_coords.shape, ss_coords_src.shape, ss_coords_dst.shape

In [ ]:
ss_view = TRELLIS2SparseStructureView(
    coords=ss_coords,
    grid_size=SS_LATENT_RESOLUTION,
    batch_size=1,
)

In [ ]:
torch.cuda.empty_cache()

ss_noise = ss_noise_spg_sampler.sample(
    batch_size=1,
    grid_size=SS_LATENT_RESOLUTION,
    feat_dim=SS_FEAT_DIM,
    seed=SEED,
    device=DEVICE,
    projector=ss_projector,
    to_sparse_view=ss_view.to_sparse_view,
    to_original_view=ss_view.to_original_view,
    self_include=True,
)

ss_flow_model.to(DEVICE)
ss_traj = flow_solver.sample(
    noise=ss_noise,
    predictor=ss_flow_spg_predictor,
    steps=TRELLIS2_SPARSE_STRUCTURE_STEPS,
    predictor_args={
        "cond": cond,
        "neg_cond": neg_cond,
        "projector": ss_projector,
        "to_sparse_view": ss_view.to_sparse_view,
        "to_original_view": ss_view.to_original_view,
        "self_include": True,
    },
    sigma_min=SIGMA_MIN,
    rescale_t=TRELLIS2_SPARSE_STRUCTURE_RESCALE_T,
    verbose=True,
    tqdm_desc="Sampling sparse structure",
)
ss_flow_model.cpu()
torch.cuda.empty_cache()

In [ ]:
ss_decoder.to(DEVICE)
with torch.no_grad():
    ss_occ_logits = ss_decoder(ss_traj[-1].x_t.to(DEVICE))
ss_occ = ss_occ_logits > 0
coords_symm = trellis2_sparse_structure_logits_to_coords(
    logits=ss_occ_logits,
    target_resolution=SS_TARGET_RESOLUTION,
)
ss_decoder.cpu()
torch.cuda.empty_cache()

ss_occ.shape, coords_symm.shape

In [ ]:
voxel_mesh = voxel_grid_to_cube_mesh(
    ss_occ[0, 0].detach().cpu().numpy(),
    grid_res=ss_occ.shape[-1],
)

voxel_mesh.show()

In [ ]:
shape_coords_src, shape_coords_dst, shape_rows_src, shape_rows_dst, shape_O, shape_t, shape_s = build_symmetry_relation_inputs(
    coords=coords_symm,
    relations=relations,
    grid_size=SHAPE_LATENT_RESOLUTION,
)

shape_mapper.to(DEVICE)
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    shape_coeff = shape_mapper(
        coords_src=shape_coords_src,
        coords_dst=shape_coords_dst,
        O_dst2src=shape_O,
        t_dst2src=shape_t,
        s_dst2src=shape_s,
    )
shape_coeff = shape_coeff.to(device=torch.device(DEVICE), dtype=torch.float32)
shape_mapper.cpu()

shape_projector = SymmetryProjector(
    num_rows=coords_symm.shape[0],
    rows_src=shape_rows_src,
    rows_dst=shape_rows_dst,
    coeff=shape_coeff,
)

coords_symm.shape, shape_coords_src.shape, shape_coords_dst.shape

In [ ]:
shape_view = TRELLIS2ShapeLatentView(
    coords=coords_symm,
    sp_class=SparseTensor,
)

In [ ]:
shape_noise = shape_noise_spg_sampler.sample(
    sp_class=SparseTensor,
    coords=coords_symm,
    feat_dim=SHAPE_FEAT_DIM,
    grid_size=SHAPE_LATENT_RESOLUTION,
    seed=SEED,
    device=DEVICE,
    projector=shape_projector,
    to_sparse_view=shape_view.to_sparse_view,
    to_original_view=shape_view.to_original_view,
    self_include=True,
)

shape_flow_model.to(DEVICE)
shape_traj = flow_solver.sample(
    noise=shape_noise,
    predictor=shape_flow_spg_predictor,
    steps=TRELLIS2_SHAPE_LATENT_STEPS,
    predictor_args={
        "cond": cond,
        "neg_cond": neg_cond,
        "projector": shape_projector,
        "to_sparse_view": shape_view.to_sparse_view,
        "to_original_view": shape_view.to_original_view,
        "self_include": True,
    },
    sigma_min=SIGMA_MIN,
    rescale_t=TRELLIS2_SHAPE_LATENT_RESCALE_T,
    verbose=True,
    tqdm_desc="Sampling shape latent",
)
shape_flow_model.cpu()
torch.cuda.empty_cache()

In [ ]:
shape_latent = shape_traj[-1].x_t
shape_slat = SparseTensor(
    feats=shape_view.to_sparse_view(shape_latent),
    coords=coords_symm,
)

shape_decoder.to(DEVICE)
with torch.no_grad():
    shape_meshes, shape_subs = shape_decoder(shape_slat, return_subs=True)
shape_decoder.cpu()
torch.cuda.empty_cache()

shape_mesh = shape_meshes[0]
shape_mesh.vertices.shape, shape_mesh.faces.shape

In [ ]:
glb, verts, faces = clean_trellis2_mesh_output(
    shape_mesh=shape_mesh,
    res=SHAPE_RESOLUTION,
    remesh=True,
    simplify=True,
    decimation_target=500000,
)

OUTPUT_GLB_PATH.parent.mkdir(exist_ok=True, parents=True)
glb.export(OUTPUT_GLB_PATH)
OUTPUT_GLB_PATH

In [ ]:
glb.show()